# 17: Simple RNN - Networks with Memory

## The Recurrent Connection

**Recurrent Neural Networks (RNNs)** solve the sequence problem by adding **memory**:
- Process one element at a time
- Maintain a **hidden state** (memory)
- Pass this state to the next step

### The Web Dev Analogy

RNNs are like **Redux state management**:
- Each action updates the state
- State persists across actions
- Current state depends on all previous actions
- State + new input → new state

## What You'll Learn
- [ ] Implement a simple RNN cell with a hidden state
- [ ] Explain how recurrence processes sequences one step at a time
- [ ] Trace information flow through time steps and verify hidden state shapes

## Connection to Previous Lessons

| What you learned | How it connects here |
|-----------------|---------------------|
| **Lesson 16**: Word order matters | RNN processes tokens *one at a time*, maintaining a hidden state that remembers the past |
| **Lesson 8**: Neural network layers | An RNN cell is just a layer that feeds its output back as input at the next step |

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
torch.manual_seed(42)

print("Ready to build RNNs! 🔄")

## 1. The RNN Formula

At each time step `t`:

```
h_t = tanh(W_hh * h_(t-1) + W_xh * x_t + b)
```

Where:
- `h_t` = hidden state at time t (the memory)
- `x_t` = input at time t
- `W_hh` = weights for previous hidden state
- `W_xh` = weights for current input
- `b` = bias

**Key insight**: Same weights used at every time step!

In [ ]:
# Manual RNN implementation
class SimpleRNN:
    def __init__(self, input_size, hidden_size):
        # Initialize weights
        self.hidden_size = hidden_size
        
        # W_xh: input to hidden
        self.W_xh = np.random.randn(input_size, hidden_size) * 0.01
        
        # W_hh: hidden to hidden (the recurrent connection!)
        self.W_hh = np.random.randn(hidden_size, hidden_size) * 0.01
        
        # Bias
        self.b = np.zeros((1, hidden_size))
        
    def step(self, x_t, h_prev):
        """Single RNN step."""
        # h_t = tanh(W_xh * x_t + W_hh * h_prev + b)
        h_t = np.tanh(
            np.dot(x_t, self.W_xh) +      # Input contribution
            np.dot(h_prev, self.W_hh) +   # Previous hidden state (memory!)
            self.b
        )
        return h_t
    
    def forward(self, inputs):
        """Process entire sequence."""
        # inputs: (seq_len, input_size)
        seq_len = inputs.shape[0]
        
        # Initialize hidden state
        h = np.zeros((1, self.hidden_size))
        
        # Store all hidden states
        hidden_states = []
        
        # Process sequence
        for t in range(seq_len):
            x_t = inputs[t:t+1]  # Get input at time t
            h = self.step(x_t, h)  # Update hidden state
            hidden_states.append(h.copy())
        
        return np.array(hidden_states)

# Test it
rnn = SimpleRNN(input_size=3, hidden_size=5)

# Sequence of 4 time steps, each with 3 features
sequence = np.random.randn(4, 3)

hidden_states = rnn.forward(sequence)

print(f"Input shape: {sequence.shape}")
print(f"Output shape: {hidden_states.shape}")
print(f"\nHidden states at each time step:")
for t, h in enumerate(hidden_states):
    print(f"  t={t}: {h.flatten()[:3].round(3)}... (showing first 3 values)")

`★ Insight ─────────────────────────────────────`

**Why RNNs are special:**
1. **Same weights at every step** - parameter sharing across time
2. **Hidden state flows forward** - each step influences all future steps
3. **Variable length sequences** - can process any length input

`─────────────────────────────────────────────────`

## 2. PyTorch RNN

In [ ]:
# PyTorch makes it easy!
input_size = 10   # Embedding dimension
hidden_size = 20  # Hidden state dimension
num_layers = 1    # Number of stacked RNN layers

rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

print(f"RNN Architecture:")
print(f"  Input size: {input_size}")
print(f"  Hidden size: {hidden_size}")
print(f"  Num layers: {num_layers}")

# Example: batch of 2 sequences, each 5 time steps, each with 10 features
batch_size = 2
seq_len = 5

x = torch.randn(batch_size, seq_len, input_size)
print(f"\nInput shape: {x.shape}")
print(f"  (batch_size, seq_len, input_size)")

# Forward pass
output, h_n = rnn(x)

print(f"\nOutput shape: {output.shape}")
print(f"  (batch_size, seq_len, hidden_size)")
print(f"  → Hidden state at EVERY time step")

print(f"\nFinal hidden shape: {h_n.shape}")
print(f"  (num_layers, batch_size, hidden_size)")
print(f"  → Hidden state at LAST time step only")

## 3. Simple Sequence Classification

In [ ]:
# Classify sequences: sum > 0 or sum <= 0?
# Generate data
def generate_sequences(n_samples, seq_len):
    X = torch.randn(n_samples, seq_len, 1)
    y = (X.sum(dim=1) > 0).float().squeeze()
    return X, y

train_X, train_y = generate_sequences(1000, seq_len=10)
test_X, test_y = generate_sequences(200, seq_len=10)

print(f"Training data: {train_X.shape}, {train_y.shape}")
print(f"Test data: {test_X.shape}, {test_y.shape}")

# Example
print(f"\nExample sequence: {train_X[0].flatten()[:5].tolist()}...")
print(f"Sum: {train_X[0].sum().item():.3f}")
print(f"Label: {'Positive' if train_y[0] == 1 else 'Negative'}")

In [ ]:
# Build classifier
class RNNClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
        
    def forward(self, x):
        # x: (batch, seq_len, input_size)
        output, h_n = self.rnn(x)
        
        # Use final hidden state for classification
        # h_n: (1, batch, hidden_size) -> (batch, hidden_size)
        final_hidden = h_n.squeeze(0)
        
        # Classification
        logits = self.fc(final_hidden)
        return logits

model = RNNClassifier(input_size=1, hidden_size=32, output_size=1)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(model)

In [ ]:
# --- Training loop: feed data, compute loss, update weights ---
def train_rnn(model, train_X, train_y, test_X, test_y, n_epochs=50, batch_size=32):
    """Train the RNN classifier and track metrics."""
    history = {'train_loss': [], 'train_acc': [], 'test_acc': []}

    for epoch in range(n_epochs):
        model.train()
        epoch_loss = 0
        correct = 0

        # Mini-batch training
        for i in range(0, len(train_X), batch_size):
            batch_X = train_X[i:i+batch_size]
            batch_y = train_y[i:i+batch_size].unsqueeze(1)

            # Forward: pass sequences through RNN
            outputs = model(batch_X)

            # Compute how wrong the predictions are
            loss = criterion(outputs, batch_y)

            # Backward: compute gradients
            loss.backward()

            # Clip gradients to prevent explosion (RNNs are prone to this!)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            # Update weights using gradients
            optimizer.step()
            optimizer.zero_grad()

            epoch_loss += loss.item()
            preds = (torch.sigmoid(outputs) > 0.5).float()
            correct += (preds == batch_y).sum().item()

        train_loss = epoch_loss / (len(train_X) // batch_size)
        train_acc = correct / len(train_X)

        # Evaluate on test set
        model.train(False)
        with torch.no_grad():
            test_outputs = model(test_X)
            test_preds = (torch.sigmoid(test_outputs) > 0.5).float()
            test_acc = (test_preds.squeeze() == test_y).float().mean().item()

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_acc'].append(test_acc)

        if epoch % 10 == 0:
            print(f"Epoch {epoch:3d}: Loss={train_loss:.4f}, "
                  f"Train Acc={train_acc:.2%}, Test Acc={test_acc:.2%}")

    return history

print("✓ Training function defined. Ready to train!")

In [ ]:
# --- Train the RNN! ---
print("Training RNN on sequence classification...")
print("Watch loss decrease and accuracy increase:\n")

# Run training (uses the function defined above)
history = train_rnn(model, train_X, train_y, test_X, test_y, n_epochs=50)

print(f"\n✓ Training complete!")
print(f"  Final train accuracy: {history['train_acc'][-1]:.1%}")
print(f"  Final test accuracy:  {history['test_acc'][-1]:.1%}")

In [ ]:
# Visualize training
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['train_loss'])
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['train_acc'], label='Train')
axes[1].plot(history['test_acc'], label='Test')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 The RNN learned to aggregate information across the sequence!")

## 4. Visualizing Hidden States

In [ ]:
# Extract hidden states for a single sequence
model.train(False)

# Use a test sequence
test_seq = test_X[0:1]  # Shape: (1, 10, 1)
test_label = test_y[0].item()

# Get all hidden states
with torch.no_grad():
    output, _ = model.rnn(test_seq)
    # output: (1, 10, 32) - hidden state at each time step
    
hidden_states = output.squeeze(0).numpy()  # (10, 32)
input_values = test_seq.squeeze().numpy()  # (10, 1)

print(f"Input sequence: {input_values.flatten()[:5].round(3).tolist()}...")
print(f"True label: {'Positive' if test_label == 1 else 'Negative'}")
print(f"\nHidden states shape: {hidden_states.shape}")
print(f"  10 time steps, each with 32-dim hidden state")

In [ ]:
# Visualize how hidden state evolves
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Plot input values
axes[0].plot(input_values, 'o-', linewidth=2, markersize=8)
axes[0].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Time Step')
axes[0].set_ylabel('Input Value')
axes[0].set_title(f'Input Sequence (sum={input_values.sum():.2f})')
axes[0].grid(True, alpha=0.3)

# Plot hidden state evolution (first 8 dimensions)
for i in range(8):
    axes[1].plot(hidden_states[:, i], alpha=0.6, label=f'h{i}')

axes[1].set_xlabel('Time Step')
axes[1].set_ylabel('Hidden State Value')
axes[1].set_title('Hidden State Evolution (first 8 dimensions)')
axes[1].legend(loc='right', bbox_to_anchor=(1.15, 0.5), ncol=1)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Watch how hidden states change as the sequence is processed!")
print("   Each new input updates the hidden state.")

## 5. The Vanishing Gradient Problem

RNNs have a critical weakness: **vanishing gradients**

In [ ]:
# Demonstrate vanishing gradients
# Create sequences where the first element determines the label
def generate_long_range(n_samples, seq_len):
    """First element determines label, rest is noise."""
    X = torch.randn(n_samples, seq_len, 1)
    # Label based ONLY on first element
    y = (X[:, 0, 0] > 0).float()
    return X, y

# Test with different sequence lengths
seq_lengths = [5, 10, 20, 50]
results = []

for seq_len in seq_lengths:
    print(f"\nTesting seq_len={seq_len}...")
    
    # Generate data
    train_X, train_y = generate_long_range(1000, seq_len)
    test_X, test_y = generate_long_range(200, seq_len)
    
    # Train simple RNN
    model = RNNClassifier(input_size=1, hidden_size=32, output_size=1)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    
    # Quick training
    for epoch in range(30):
        outputs = model(train_X)
        loss = criterion(outputs, train_y.unsqueeze(1))
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    
    # Test
    model.train(False)
    with torch.no_grad():
        test_outputs = model(test_X)
        test_preds = (torch.sigmoid(test_outputs) > 0.5).float()
        test_acc = (test_preds.squeeze() == test_y).float().mean().item()
    
    results.append(test_acc)
    print(f"  Test Accuracy: {test_acc:.2%}")

# Visualize
plt.figure(figsize=(10, 6))
plt.plot(seq_lengths, results, 'o-', linewidth=2, markersize=10)
plt.axhline(y=0.5, color='red', linestyle='--', label='Random guessing')
plt.xlabel('Sequence Length')
plt.ylabel('Test Accuracy')
plt.title('RNN Performance Degrades with Longer Sequences')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("\n❌ RNNs struggle with long-range dependencies!")
print("   Gradients vanish as they backpropagate through time.")
print("   Solution: LSTM and GRU! →")

`★ Insight ─────────────────────────────────────`

**Why gradients vanish in RNNs:**
1. **Many multiplications** - gradient flows through many time steps
2. **tanh saturation** - tanh derivative is small (< 1)
3. **Small numbers multiplied** → exponentially smaller → vanishes

This prevents RNNs from learning long-range dependencies!

`─────────────────────────────────────────────────`

## 📝 Check Your Understanding

1. What is the hidden state in an RNN?
2. How does an RNN process variable-length sequences?
3. Why do we use the same weights at every time step?
4. What is the vanishing gradient problem?
5. When do RNNs struggle most?

In [ ]:
# --- Exercise: Break the RNN (modify & observe) ---
# What happens when hidden_size is too small?

# Create a TINY RNN (hidden_size=2 instead of 32)
tiny_rnn = RNNClassifier(input_size=1, hidden_size=2, output_size=1)
tiny_criterion = nn.BCEWithLogitsLoss()
tiny_optimizer = torch.optim.Adam(tiny_rnn.parameters(), lr=0.001)

# Re-generate training data (same as before)
tiny_train_X, tiny_train_y = generate_sequences(1000, seq_len=10)
tiny_test_X, tiny_test_y = generate_sequences(200, seq_len=10)

tiny_losses = []
for epoch in range(50):
    tiny_rnn.train()
    outputs = tiny_rnn(tiny_train_X)
    loss = tiny_criterion(outputs, tiny_train_y.unsqueeze(1))
    tiny_optimizer.zero_grad()
    loss.backward()
    tiny_optimizer.step()
    tiny_losses.append(loss.item())

# Check accuracy
tiny_rnn.train(False)
with torch.no_grad():
    tiny_preds = (torch.sigmoid(tiny_rnn(tiny_test_X)) > 0.5).float()
    tiny_acc = (tiny_preds.squeeze() == tiny_test_y).float().mean().item()

print(f"Normal RNN (hidden=32): {history['test_acc'][-1]:.1%} accuracy")
print(f"Tiny RNN   (hidden=2):  {tiny_acc:.1%} accuracy")
print()
if tiny_acc < 0.7:
    print("❌ Tiny RNN struggles! Not enough memory to represent the pattern.")
else:
    print("Hmm, tiny RNN still works — try an even harder task!")
print()
print("💡 hidden_size = how much the RNN can 'remember'")
print("   Too small → can't learn complex patterns")
print("   Too large → overfits and wastes computation")

In [ ]:
# --- Exercise 1: One RNN Step ---
# Compute one step of a simple RNN manually.
# h_new = tanh(x @ W_xh + h_prev @ W_hh + b_h)

h_prev = np.array([0.5, -0.3])           # Previous hidden state (size 2)
x = np.array([1.0])                       # Current input (size 1)
W_xh = np.array([[0.1], [0.2]]).T        # (1, 2) → maps input to hidden
W_hh = np.array([[0.3, 0.1], [0.1, 0.4]])  # (2, 2) → maps hidden to hidden
b_h = np.array([0.0, 0.0])               # Bias

# YOUR CODE HERE:
h_new = None  # Compute: tanh(x @ W_xh + h_prev @ W_hh + b_h)

# --- Check ---
assert h_new is not None, "Compute the new hidden state!"
assert h_new.shape == (2,), f"Hidden state should have shape (2,), got {h_new.shape}"
assert all(-1 <= v <= 1 for v in h_new), "tanh output should be between -1 and 1"
expected = np.tanh(x @ np.array([[0.1, 0.2]]) + h_prev @ W_hh + b_h).flatten()
assert np.allclose(h_new, expected, atol=0.01), f"Values don't match expected: {expected.round(3)}"
print(f"Exercise 1 passed! ✓  (h_new = {h_new.round(3)})")

# --- Quick Check: Hidden State ---
# What does the hidden state in an RNN represent?
# a) The raw input at the current time step
# b) A summary/memory of all previous tokens processed so far
# c) The final output prediction
# d) The learning rate

your_answer = None  # Put 'a', 'b', 'c', or 'd'

# --- Check ---
assert your_answer is not None, "Pick an answer!"
assert your_answer == 'b', "The hidden state accumulates information from all previous steps — it's the RNN's 'memory'!"
print("Exercise 2 passed! ✓")

print("\n🎉 All exercises passed!")

## 🎯 Summary

**RNNs add memory to neural networks**:
- Hidden state `h_t` serves as memory
- Process sequences step by step
- Same weights used at every time step

**Key operations**:
- `nn.RNN(input_size, hidden_size)` - Create RNN layer
- Returns all hidden states + final state
- Use final state for classification

**Critical limitation**:
- **Vanishing gradients** prevent learning long-range dependencies
- Performance degrades with sequence length

**Next up**: LSTM and GRU solve this problem! →